# Cvičení 1: Barvy a segmentace podle barvy

Cílem cvičení je vyzkoušet si práci s barevnými prostory a tři jednoduché způsoby segmentace objektu podle barvy. Objektem bude tenisový míček na obrázku `data/tennis_ball2_small.jpg`.

1. Načtení obrázku
2. Segmentace prahováním v prostoru RGB
3. Segmentace prahováním v prostoru HSV
4. Segmentace zpětnou projekcí histogramu

Obecné pokyny:

- Můžete používat [OpenCV](https://docs.opencv.org/) nebo např. [scikit-image](https://scikit-image.org/).
- Výsledky vykreslujte pomocí [matplotlib](https://matplotlib.org/).
- Svůj kód pište na místa označená komentářem `# TODO: implement`.
- U každé segmentace (body 2 až 4) navíc spočítejte a vykreslete **bounding box** a **těžiště** míčku.
- Základem všech tří segmentací je prahování. Morfologické operace ani jiné následné úpravy masky nejsou vyžadovány, můžete je ale použít pro vylepšení výsledků.
- Každá sekce obsahuje vzorový výstup. Váš výsledek nemusí být na pixel stejný, ale měl by vypadat podobně.
- Za každým bodem je buňka s testy ze souboru `tests/test_colors.py`. Testy porovnávají vaši masku, bounding box a těžiště s ručně vytvořenou referenční segmentací míčku.
  - Masky a bounding boxy se porovnávají pomocí **Jaccardova indexu** (IoU, intersection over union)
    $$
    \mathrm{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}
    $$
    kde $A$ je váš výsledek a $B$ reference. Hodnota 1 znamená dokonalou shodu, hodnota 0 žádný překryv.
  - U těžiště se měří vzdálenost od referenčního těžiště v pixelech.
  - Požadované hodnoty jsou uvedené u každé úlohy.

In [ ]:
import sys

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

if ".." not in sys.path:
    sys.path.append("..")
from tests import test_colors

## 1. Načtení obrázku

- Načtěte obrázek `../data/tennis_ball2_small.jpg`.
- OpenCV načítá kanály v pořadí **BGR**, matplotlib očekává **RGB**. Převeďte obrázek do RGB a uložte jej do proměnné `rgb`.
- Vypište rozměry (`shape`) a datový typ (`dtype`) obrázku.
- Vykreslete vedle sebe barevný obrázek a jeho jednotlivé kanály R, G a B v odstínech šedi. Kanály se budou hodit při volbě prahů v dalším bodě.

Vzorový výstup:

<figure class="image">
  <img src="../figures/colors-expected_outputs_load.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement

# rgb = ...

########################################

In [ ]:
test_colors.TestLoadImage.eval(rgb=rgb)

## Pomocná funkce: bounding box a těžiště

Bounding box a těžiště budeme potřebovat ve všech třech segmentačních úlohách, proto si je připravíme předem jako funkci.

- `bbox_and_centroid(mask)`
  - Vstupem je binární maska (`uint8`, hodnoty 0 a 255).
  - Vrátí bounding box všech nenulových pixelů jako čtveřici `(x, y, w, h)` a těžiště jako dvojici `(cx, cy)` v pixelech.
  - Těžiště spočítejte z obrazových momentů: $c_x = m_{10} / m_{00}$, $c_y = m_{01} / m_{00}$.

In [ ]:
def bbox_and_centroid(mask: np.ndarray) -> tuple[tuple[int, int, int, int], tuple[float, float]]:
    """
    Compute the bounding box and the centroid of a binary mask.

    Args:
        mask: Binary mask of dtype uint8 with values 0 and 255.
    Returns:
        Bounding box (x, y, w, h) and centroid (cx, cy) in pixels.
    """
    ########################################
    # TODO: implement

    raise NotImplementedError

    ########################################

In [ ]:
test_colors.TestBboxAndCentroid.eval(bbox_and_centroid_fn=bbox_and_centroid)

## 2. Segmentace prahováním v prostoru RGB

- Stejná barva má na osvětlené a zastíněné straně míčku velmi odlišné hodnoty R, G a B, takže prahování původních kanálů nefunguje dobře. Vliv jasu lze potlačit přechodem na **relativní zastoupení** jednotlivých složek
  $$
  r = \frac{R}{R + G + B}, \qquad g = \frac{G}{R + G + B}, \qquad b = \frac{B}{R + G + B}
  $$
  - Před dělením převeďte obrázek na desetinná čísla a ošetřete dělení nulou u černých pixelů.
- Zvolte prahy relativních složek tak, aby maska pokrývala míček a co nejméně pozadí.
  - Míček je žlutozelený, má tedy vysoký podíl zelené složky. Červené pozadí i kůže ruky mají podíl zelené výrazně nižší.
- **Nápověda:** samotné relativní složky nestačí. U tmavých pixelů je součet $R + G + B$ malý, relativní složky jsou proto zašuměné a v tmavém pozadí vznikají falešné detekce. Prahujte proto zároveň i **jas**, např. součet $R + G + B$, a tmavé pixely z masky vyřaďte.
- Vytvořte binární masku typu `uint8` s hodnotami 0 a 255 a uložte ji do proměnné `mask_rgb`.
- Spočítejte bounding box a těžiště masky (`bbox_rgb`, `centroid_rgb`) a vypište je.
- Vykreslete trojici obrázků: původní RGB, binární masku a RGB s vyznačeným bounding boxem a těžištěm pomocí `ax.plot`.
- Vyzkoušejte si prahování bez podmínky na jas. Co se stane s bounding boxem a co s těžištěm? Proč je bounding box na osamocené falešné pixely mnohem citlivější?
- Požadavky testů: IoU masky alespoň **0,75**, IoU bounding boxu alespoň **0,85**, těžiště nejvýše **20 px** od reference.

Vzorový výstup:

<figure class="image">
  <img src="../figures/colors-expected_outputs_rgb.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement



########################################

In [ ]:
test_colors.TestSegmentationRGB.eval(mask=mask_rgb, bbox=bbox_rgb, centroid=centroid_rgb)

## 3. Segmentace prahováním v prostoru HSV

- Převeďte obrázek do prostoru HSV. Výsledek uložte do proměnné `hsv`.
  - Pozor: pro 8bitové obrázky má v OpenCV kanál H rozsah **0 až 179** (úhel ve stupních dělený dvěma), kanály S a V mají rozsah 0 až 255.
- Barvu míčku vymezte pomocí kanálu **H** (odstín). Zvolte dolní a horní mez odstínu tak, aby odpovídaly žlutozelené barvě míčku.
  - Pomůže histogram kanálu H nebo jeho vykreslení jako obrázku.
- **Nápověda:** samotný odstín nestačí. U tmavých pixelů je odstín nespolehlivý a v tmavém pozadí vznikají falešné detekce. Prahujte proto zároveň i **jas** (kanál V) a tmavé pixely z masky vyřaďte. Kanál S nepoužívejte.
- Vytvořte binární masku typu `uint8` s hodnotami 0 a 255 a uložte ji do proměnné `mask_hsv`.
- Spočítejte bounding box a těžiště masky (`bbox_hsv`, `centroid_hsv`) a vypište je.
- Vykreslete trojici obrázků: původní RGB, binární masku a RGB s vyznačeným bounding boxem a těžištěm pomocí `ax.plot`.
- Požadavky testů: IoU masky alespoň **0,75**, IoU bounding boxu alespoň **0,75**, těžiště nejvýše **20 px** od reference.
- Porovnejte výsledek s prahováním v RGB. Který prostor se pro volbu prahů hodí lépe a proč? Která část míčku v masce chybí?

Vzorový výstup:

<figure class="image">
  <img src="../figures/colors-expected_outputs_hsv.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement



########################################

In [ ]:
test_colors.TestSegmentationHSV.eval(mask=mask_hsv, bbox=bbox_hsv, centroid=centroid_hsv)

## 4. Segmentace zpětnou projekcí histogramu

Místo ručního ladění prahů si barvu míčku „naučíme“ z ukázky: z vybrané části míčku spočítáme referenční histogram a pro každý pixel obrázku pak zjistíme, jak často se jeho barva v ukázce vyskytovala.

- Ručně zadejte souřadnice obdélníku `(x, y, w, h)`, který leží celý uvnitř míčku. Obdélník by měl obsahovat co nejvíce odstínů míčku, ale ne šev ani nápis.
- Z výřezu spočítejte jednorozměrný histogram **pouze kanálu H** (odstín) v prostoru HSV.
  - Kanál H má v OpenCV rozsah 0 až 179, použijte tedy 180 binů.
- Histogram normalizujte do rozsahu 0 až 255 a uložte do proměnné `hist`.
- Spočítejte zpětnou projekci histogramu na celý obrázek a uložte ji do proměnné `backproj`.
- Zpětnou projekci naprahujte na binární masku typu `uint8` s hodnotami 0 a 255 a uložte ji do proměnné `mask_bp`.
- **Nápověda:** zpětná projekce odstínu trpí u tmavých pixelů stejně jako prahování odstínu v bodě 3. I zde proto z masky vyřaďte pixely s nízkým **jasem** (kanál V).
- Spočítejte bounding box a těžiště masky (`bbox_bp`, `centroid_bp`) a vypište je.
- Vykreslete čtveřici obrázků: původní RGB s vyznačeným referenčním obdélníkem, zpětná projekce, binární maska, RGB s bounding boxem a těžištěm.
- Histogram malého výřezu nepokryje všechny odstíny míčku, proto jsou požadavky testů nižší: IoU masky alespoň **0,7**, IoU bounding boxu alespoň **0,7**, těžiště nejvýše **30 px** od reference.
- Která část míčku ve výsledku chybí a proč? Čím by šlo metodu vylepšit?

Vzorový výstup:

<figure class="image">
  <img src="../figures/colors-expected_outputs_backprojection.png" alt="Vzorový výstup" style="width: 100%"/>
</figure>

In [ ]:
########################################
# TODO: implement



########################################

In [ ]:
test_colors.TestHueHistogram.eval(hist=hist)

In [ ]:
test_colors.TestBackprojection.eval(backproj=backproj)

In [ ]:
test_colors.TestSegmentationBackprojection.eval(mask=mask_bp, bbox=bbox_bp, centroid=centroid_bp)